# Titanic - Machine Learning from Disaster

## 1. Importing necessary libraries and loading the dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', None)
%matplotlib inline

In [ ]:
train = pd.read_csv('/kaggle/input/titanic/train.csv')
test = pd.read_csv('/kaggle/input/titanic/test.csv')

train.head()

## 2. Exploratory Data Analysis

Let's do some quick analysis of our dataset

In [ ]:
train.shape

very small dataset, also the test set doesn't contain target labels that's why cross validation is a good option to evaluate the performances of models to be trained

In [ ]:
train.columns

In [ ]:
train.info()

In [ ]:
train.describe()

In [ ]:
train.isnull().sum()

There are many missing values in the Cabin column and some in the Age column

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.histplot(train['Age'], ax=axes[0])
axes[0].set_xlabel('Age values')
axes[0].set_ylabel('Frequency')

sns.boxplot(train['Age'], ax=axes[1])
axes[1].set_title('Boxplot for Age')

plt.tight_layout()
plt.show()

The 'Age' variable distribution is right-skewed, so mean imputing can cause problems as it's sensitive to outliers - applying median instead will be a better option. Instead of imputing right here, I will fill in nulls after some feature engineering 

In [ ]:
miss = (train['Cabin'].isna().sum())/len(train['Cabin'])
print(f'Cabin missing {round(miss*100, 2)}% of values')

In [ ]:
print(train['Cabin'].unique())
print(train['Cabin'].nunique(), 'unique values')

although more than half of the values are missing and the column contains 147 unique values, 'it can still give us important insights, I will keep it and create a separate class with missing values, later the cabin letter (deck) will be extracted as a separate feature

In [ ]:
train.fillna({"Cabin":"U"}, inplace=True) #U for the Unknown

since the dataset is small, every bit of data is needed, for the 'Embarked' with 2 missing values I will impute those with the mode

In [ ]:
mode = train['Embarked'].mode()[0] #more than 1 modes possible, therefore use the indexing to choose one
train.fillna({"Embarked": mode}, inplace=True)

In [ ]:
train.isna().sum()

All the missing values are addressed, let's check for duplicates

In [ ]:
train['PassengerId'].duplicated().sum()

In [ ]:
train.describe(include='object')

Assuming that family members shared the same ticket (as we have only 681 unique tickets out of 891), we can create a new binary feature to indicate if a passenger had a **shared** ticket or no apart form the 'TicketGroupSize'

## 3. Feature Engineering

In [ ]:
ticket_counts = train['Ticket'].value_counts()
train['TicketGroupSize'] = train['Ticket'].map(ticket_counts)
train.head()

In [ ]:
train['SharedTicket'] = (train['TicketGroupSize']>1).astype(int)
train.head()

Also, it might be the case that passengers with bigger families had lower or higher chances of survival than passengers who were travelling alone, let's create a feature 'FamilySize' feature

In [ ]:
train['FamilySize'] = train['SibSp'] + train['Parch'] + 1
train.head()

As mentioned earlier, let's create a new feature using the first letter form the 'Cabin' column which means 'Deck'

In [ ]:
train['Deck'] = train['Cabin'].str[0]

In [ ]:
train.tail()

good, now we have a new feature named 'Deck', let's also try to extract titles from the names

In [ ]:
train['Name'].sample(5)

In [ ]:
train['Title'] = train['Name'].str.extract(r',\s*([^.]*)\.')

In [ ]:
train['Title'].unique()

Let's group the titles to have fewer categories to also capture gender + age + social status

In [ ]:
train['Title'] = train['Title'].replace({
    #Foregign titles !Ms will be also substituted by Miss as Ms became popular in the 1950s only
    'Mme': 'Mrs',
    'Ms': 'Miss',
    'Mlle': 'Miss',

    #Officer titles (military, professional, clergy)
    'Rev': 'Officer',
    'Dr': 'Officer',
    'Major': 'Officer',
    'Col': 'Officer',
    'Capt': 'Officer',

    #Royalty/nobility titles
    'Don': 'Royalty',
    'Lady': 'Royalty',
    'Sir': 'Royalty',
    'the Countess': 'Royalty',
    'Jonkheer': 'Royalty'
})

In [ ]:
train['Title'].unique()

In [ ]:
# Median age for each Title group
title_age_median = train.groupby('Title')['Age'].median()
print(title_age_median)

Now we can impute missing values for the 'Age' column

In [ ]:
train['Age'] = train.apply(lambda row: title_age_median[row['Title']] if pd.isnull(row['Age']) else row['Age'], axis=1)
train.head()

In [ ]:
train['Age'].isnull().sum()

Now, let's also do some feature engineering using the 'Age' column, we can group passengers into categories based on their ages

In [ ]:
print(f"The passengers age range between: {train['Age'].min()} and {train['Age'].max()}")

In [ ]:
train['AgeGroup'] = pd.cut(train['Age'], bins=[0, 12, 18, 35, 60, 80], labels=['Child','Teen','Adult','Middle','Senior'])

In [ ]:
train.head()

In [ ]:
train.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'], inplace=True)

In [ ]:
train.sample(5)

## 4. Analysis through Visualizations


In [ ]:
plt.figure(figsize=(8,6))
sns.countplot(data=train, x='Survived', palette='deep')
plt.xticks([0,1], ['Not Survived', 'Survived'])
plt.xlabel('Target')
plt.title('Survived vs Non-survived')
plt.show()

the dataset is imbalanced, during the evaluation we can't rely on accuracy only

In [ ]:
survivors = train[train['Survived']==1]
fm_counts = survivors['Sex'].value_counts()

In [ ]:
plt.figure(figsize=(8, 6))
plt.pie(fm_counts, labels=fm_counts.index, autopct='%1.1f%%', startangle=140, colors=['skyblue', 'pink'])
plt.title('Male VS Female Survivors')
plt.show()

mostly women survived

In [ ]:
corr = train.corr(numeric_only=True)

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', square=True, linewidths=0.5)
plt.title('Correlation Heatmap')
plt.show()

With the newly engineered features even, there is not much linear relationship between the target variable, so it is better to fit non-linear models, that's why I will keep features highly correlated

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

sns.countplot(data=train, x='Pclass', hue='Survived', ax=axes[0, 0])
axes[0, 0].set_xlabel('Class Values')
axes[0, 0].set_ylabel('Suvived Count')

sns.countplot(data=train, x='Embarked', hue='Survived', ax=axes[0, 1])
axes[0, 1].set_xlabel('Embarked Values')
axes[0, 1].set_ylabel('Survived Count')

sns.countplot(data=train, x='Parch', hue='Survived', ax=axes[0, 2])
axes[0, 2].set_xlabel('Family size')
axes[0, 2].set_ylabel('Survived Count')

sns.countplot(data=train, x='SibSp', hue='Survived', ax=axes[1,0])
axes[1, 0].set_xlabel('Siblings Spouse Number')
axes[1, 0].set_ylabel('Survived Count')

sns.countplot(data=train, x='AgeGroup', hue='Survived', ax=axes[1, 1])
axes[1, 1].set_xlabel('Age Group')
axes[1, 1].set_ylabel('Survived Count')

sns.countplot(data=train, x='Title', hue='Survived', ax=axes[1,2])
axes[1, 2].set_xlabel('Title Values')
axes[1, 2].set_ylabel('Survived Count')

plt.tight_layout()
plt.show()

## 5. Encoding Categorical Columns

In [ ]:
train['Sex'] = train['Sex'].map( {'male': 0, 'female': 1,} ).astype(int)

train.head()

In [ ]:
embarked_encoded = pd.get_dummies(train['Embarked'], prefix='Port', dtype=int)
train = pd.concat([train, embarked_encoded], axis=1)

In [ ]:
train['Deck'].unique()

lots of categories for 'Deck' column, one-hot encoding will increase the dimensionality, so I will use label encoding (safe for Tree-based models)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le_deck = LabelEncoder()
train['Deck_Encoded'] = le_deck.fit_transform(train['Deck'])

In [ ]:
le_title = LabelEncoder()
train['Title_Encoded'] = le_title.fit_transform(train['Title'])

In [ ]:
le_age_group = LabelEncoder()
train['AgeGroup_Encoded'] = le_age_group.fit_transform(train['AgeGroup'])

In [ ]:
train.drop(columns=['Deck', 'Title', 'AgeGroup', 'Embarked'], inplace=True)
train.sample(5)

## 6. Tree-Based Model Fitting with Cross Validation

In [ ]:
X_train = train.drop(columns='Survived')
y_train = train['Survived']

In [ ]:
X_train, y_train

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

k_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)

Let's compare the tree based models using the created k_folds

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


models = {'RF': RandomForestClassifier(n_estimators=200, max_depth=5, random_state=7),
         'XGB': XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=5, use_label_encoder=False, eval_metric='logloss', random_state=7),
         'LGB': LGBMClassifier(n_estimators=200, learning_rate=0.07, max_depth=5, random_state=7, verbose=-1)}

In [ ]:
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=k_folds, scoring='accuracy')
    print(f"{name} CV accuracy: {scores.mean():.3f} ± {scores.std():0.3f}")

In [ ]:
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=k_folds, scoring='f1')
    print(f"{name} CV f1-score: {scores.mean():.3f} ± {scores.std():0.3f}")

## 7. Training the Final Model

So the winner is XGBoost with the params above, now I'll fit the model on the whole train set  

In [ ]:
xgb_classifier = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=5, use_label_encoder=False, eval_metric='logloss', random_state=7)
xgb_classifier.fit(X_train, y_train)

## 8. Let's make predictions with the trained model

But first, the test set should undergo all the preprocessing steps

In [ ]:
test.head()

In [ ]:
test.isnull().sum()

In [ ]:
test.fillna({"Cabin":"U"}, inplace=True) #U for the Unknown

In [ ]:
test['Fare'].fillna(train['Fare'].median(), inplace=True)

In [ ]:
ticket_counts = test['Ticket'].value_counts()
test['TicketGroupSize'] = test['Ticket'].map(ticket_counts)

In [ ]:
test['SharedTicket'] = (test['TicketGroupSize']>1).astype(int)

In [ ]:
test['FamilySize'] = test['SibSp'] + test['Parch'] + 1

In [ ]:
test['Deck'] = test['Cabin'].str[0]

In [ ]:
test['Deck'].unique()

In [ ]:
test['Title'] = test['Name'].str.extract(r',\s*([^.]*)\.')

In [ ]:
test['Title'].unique()

'Dona' is a title we haven't seen in the train set, we'll put it under 'Royalty'

In [ ]:
test['Title'] = test['Title'].replace({
    'Mme': 'Mrs',
    'Ms': 'Miss',
    'Mlle': 'Miss',

    'Rev': 'Officer',
    'Dr': 'Officer',
    'Major': 'Officer',
    'Col': 'Officer',
    'Capt': 'Officer',

    'Don': 'Royalty',
    'Dona': 'Royalty',
    'Lady': 'Royalty',
    'Sir': 'Royalty',
    'the Countess': 'Royalty',
    'Jonkheer': 'Royalty'})

In [ ]:
test['Age'] = test.apply(lambda row: title_age_median[row['Title']] if pd.isnull(row['Age']) else row['Age'], axis=1)

In [ ]:
test['AgeGroup'] = pd.cut(test['Age'], bins=[0, 12, 18, 35, 60, 80], labels=['Child','Teen','Adult','Middle','Senior'])

In [ ]:
test.head()

In [ ]:
test['Sex'] = test['Sex'].map( {'male': 0, 'female': 1,} ).astype(int)

In [ ]:
embarked_encoded = pd.get_dummies(test['Embarked'], prefix='Port', dtype=int)
test = pd.concat([test, embarked_encoded], axis=1)

In [ ]:
test['Deck_Encoded'] = le_deck.transform(test['Deck'])
test['Title_Encoded'] = le_title.transform(test['Title'])
test['AgeGroup_Encoded'] = le_age_group.transform(test['AgeGroup'])

In [ ]:
test.head()

In [ ]:
test.columns

In [ ]:
X_test = test.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin', 'Embarked', 'Deck', 'Title', 'AgeGroup'])
X_test.head()

In [ ]:
predictions = xgb_classifier.predict(X_test)

In [ ]:
submission = pd.DataFrame({'PassengerId': test['PassengerId'], 'Survived': predictions.astype(int)})
submission.to_csv('/kaggle/working/submission.csv', index=False)